# Resume Hindi OCR v2 (CNN + Bi-LSTM + Luong attention), epochs 13 to 20

Built from your original notebook `notebook8288431bd9`: same model, tokenizer, data split, transforms, loss, optimizer groups, warmup+cosine schedule and scheduled sampling. Nothing is guessed any more.

**Before running:** attach two inputs (the weights dataset `dataset_ocr` and the Hindi OCR dataset), use **one GPU** (the original used a single GPU), and run cells top to bottom.

- Cell 7 loads your weights and runs a **safety gate**: it re-measures the first 200 validation images. You should see Acc about 0.82 and CER about 0.0145 (your epoch 12 numbers). If it stops, nothing is trained.
- Training saves a full resume state every 1000 steps. If the session dies, add this notebook's output as an input, set `RESUME_STATE_PATH`, and run again.
- Optimizer state (AdamW moments) was never saved in your original run, so the optimizer restarts fresh; a short LR re-warmup (`REWARM_STEPS`) protects the weights.

In [1]:
# ===================== SETTINGS =====================
DATA_ROOT = "/kaggle/input/datasets/prathmeshzade/hindi-ocr-synthetic-line-image-text-pair/data_80k"
IMAGE_DIR = DATA_ROOT + "/output_images/"
CSV_PATH  = DATA_ROOT + "/data.csv"

PREV_DIR       = "/kaggle/input/datasets/prachi33/dataset-ocr"
INIT_WEIGHTS   = PREV_DIR + "/latest_model"        # folder or .pth file, both handled
TOKENIZER_JSON = PREV_DIR + "/tokenizer.json"
OUTPUT_DIR     = "/kaggle/working/trocr-hi-modified-v2-resume"

# Continue THIS notebook's own earlier run: add its output as input and set e.g.
# RESUME_STATE_PATH = "/kaggle/input/notebooks/prachi33/<name>/trocr-hi-modified-v2-resume/resume_state.pt"
RESUME_STATE_PATH = None

# ---- exact values from the original notebook ----
IMG_HEIGHT, IMG_WIDTH = 128, 512
MAX_LEN       = 128
LSTM_HIDDEN, LSTM_LAYERS = 256, 2
DECODER_DIM, EMBED_DIM, ATTN_DIM = 512, 256, 256
NUM_EPOCHS    = 20
LEARNING_RATE = 3e-4
CNN_LR        = 1e-5
BATCH_SIZE    = 8
GRAD_ACCUM    = 4
WARMUP_STEPS  = 500
LABEL_SMOOTH  = 0.1
SS_WARMUP_EPOCHS = 3
SS_END_RATIO     = 0.3
EVAL_BATCH    = 8
BEAM_WIDTH    = 5

# ---- resume settings ----
START_EPOCH   = 12      # epochs already finished (0-based index of the next epoch). Epoch 13 is redone.
REWARM_STEPS  = 100     # optimizer steps of short LR ramp-up because AdamW state restarts
SAVE_EVERY_STEPS = 1000 # batches; multiple of GRAD_ACCUM
TIME_LIMIT_HOURS = 10.0 # do not start an epoch that will not fit (about 0.95 h per epoch)
GATE_MIN_ACC, GATE_MAX_CER = 0.5, 0.10
EVAL_SAMPLES  = 200     # final greedy + beam evaluation on first N test images
SEED = 42

In [2]:
import gc, os, math, random, json, time, zipfile
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from collections import Counter
from torch.utils.data import Dataset, DataLoader, Subset, Sampler
from torchvision import models
from torchvision.transforms import v2

T0 = time.time()
gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", "No GPU: set Accelerator to a GPU"
print("torch", torch.__version__, "| GPU", torch.cuda.get_device_name(0))

paths = [CSV_PATH, IMAGE_DIR, TOKENIZER_JSON, INIT_WEIGHTS]
if RESUME_STATE_PATH: paths.append(RESUME_STATE_PATH)
for p in paths:
    assert os.path.exists(p), f"Missing: {p}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("All paths OK")

torch 2.11.0+cu128 | GPU Tesla T4
All paths OK


In [3]:
df = pd.read_csv(CSV_PATH)
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() > 0].reset_index(drop=True)
split = int(len(df) * 0.9)
train_df = df[:split].reset_index(drop=True)      # same 90/10 split as the original run
test_df  = df[split:].reset_index(drop=True)
print(f"Dataset {len(df)} | train {len(train_df)} | test {len(test_df)}")

class CharTokenizer:
    PAD = "<PAD>"; UNK = "<UNK>"; BOS = "<BOS>"; EOS = "<EOS>"
    def load(self, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        self.char2idx = data["char2idx"]
        self.idx2char = {int(i): ch for ch, i in self.char2idx.items()}
        self.vocab_size = len(self.char2idx)
        self.pad_id, self.unk_id, self.bos_id, self.eos_id = 0, 1, 2, 3
        return self
    def encode(self, text, max_length=128):
        ids = [self.bos_id] + [self.char2idx.get(ch, self.unk_id) for ch in str(text)] + [self.eos_id]
        if len(ids) > max_length:
            ids = ids[:max_length - 1] + [self.eos_id]
        return ids + [self.pad_id] * (max_length - len(ids))
    def decode(self, ids, skip_special=True):
        chars = []
        for idx in ids:
            idx = idx.item() if hasattr(idx, "item") else int(idx)
            tok = self.idx2char.get(idx, "")
            if skip_special:
                if tok == self.EOS: break
                if tok in (self.PAD, self.BOS, self.UNK): continue
            chars.append(tok)
        return "".join(chars)
    def save(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"char2idx": self.char2idx}, f, ensure_ascii=False)

tokenizer = CharTokenizer().load(TOKENIZER_JSON)
assert (tokenizer.char2idx["<PAD>"], tokenizer.char2idx["<BOS>"], tokenizer.char2idx["<EOS>"]) == (0, 2, 3)
print("Vocab size:", tokenizer.vocab_size)

Dataset 80000 | train 72000 | test 8000
Vocab size: 116


In [4]:
train_transform = v2.Compose([
    v2.Resize((IMG_HEIGHT, IMG_WIDTH)),
    v2.RandomRotation(degrees=1.5),
    v2.RandomPerspective(distortion_scale=0.05, p=0.3),
    v2.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    v2.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    v2.ToImage(), v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
eval_transform = v2.Compose([
    v2.Resize((IMG_HEIGHT, IMG_WIDTH)),
    v2.ToImage(), v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class OCRDataset(Dataset):
    def __init__(self, root_dir, df, tokenizer, max_len=128, transforms=None):
        self.root_dir, self.df = root_dir, df.reset_index(drop=True)
        self.tokenizer, self.max_len, self.transforms = tokenizer, max_len, transforms
        self.valid = []
        for i in range(len(self.df)):
            text = str(self.df["text"][i]).strip()
            path = self.root_dir + self.df["image_file"][i]
            if len(text) > 0 and os.path.exists(path):
                self.valid.append(i)
        print(f"  Dataset: {len(self.valid)}/{len(df)} valid samples")
    def __len__(self): return len(self.valid)
    def __getitem__(self, idx):
        real = self.valid[idx]
        fname = self.df["image_file"][real]
        text = str(self.df["text"][real])
        try:
            image = Image.open(self.root_dir + fname).convert("RGB")
        except Exception:
            image = Image.new("RGB", (IMG_WIDTH, IMG_HEIGHT), (255, 255, 255)); text = ""
        pv = self.transforms(image)
        if not isinstance(pv, torch.Tensor): pv = pv.as_subclass(torch.Tensor)
        labels = torch.tensor(self.tokenizer.encode(text, self.max_len), dtype=torch.long)
        labels[labels == self.tokenizer.pad_id] = -100
        return {"pixel_values": pv, "labels": labels}

train_dataset = OCRDataset(IMAGE_DIR, train_df, tokenizer, MAX_LEN, train_transform)
eval_dataset  = OCRDataset(IMAGE_DIR, test_df,  tokenizer, MAX_LEN, eval_transform)
steps_per_epoch = len(train_dataset) // BATCH_SIZE
print("Steps per epoch:", steps_per_epoch, "(your log shows 9000)")
if steps_per_epoch != 9000:
    print("WARNING: differs from the original 9000 steps; LR schedule and tf positions will be slightly off")

  Dataset: 72000/72000 valid samples
  Dataset: 8000/8000 valid samples
Steps per epoch: 9000 (your log shows 9000)


In [5]:
class CNNBiLSTMEncoder(nn.Module):
    def __init__(self, hidden_size=256, num_layers=2, dropout=0.3):
        super().__init__()
        resnet = models.resnet50(weights=None)   # all weights come from your checkpoint, no download needed
        self.cnn = nn.Sequential(*list(resnet.children())[:-2])
        for i, child in enumerate(self.cnn.children()):
            for p in child.parameters():
                p.requires_grad = not (i < 6)
        self.channel_reduce = nn.Sequential(
            nn.Conv2d(2048, hidden_size * 2, kernel_size=1),
            nn.BatchNorm2d(hidden_size * 2),
            nn.ReLU(inplace=True),
        )
        self.lstm = nn.LSTM(input_size=hidden_size * 2, hidden_size=hidden_size,
                            num_layers=num_layers, batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.output_size = hidden_size * 2
        self.layer_norm = nn.LayerNorm(self.output_size)
    def train(self, mode=True):
        super().train(mode)
        for i, child in enumerate(self.cnn.children()):
            if i < 6: child.eval()
        return self
    def forward(self, x):
        feat = self.channel_reduce(self.cnn(x))
        B, C, H, W = feat.shape
        feat = feat.permute(0, 2, 3, 1).reshape(B, H * W, C)
        out, _ = self.lstm(feat)
        return self.layer_norm(out)

class MultiplicativeAttention(nn.Module):
    def __init__(self, encoder_dim, decoder_dim, attn_dim=256):
        super().__init__()
        self.query_proj = nn.Linear(decoder_dim, attn_dim, bias=False)
        self.key_proj   = nn.Linear(encoder_dim, attn_dim, bias=False)
        self.value_proj = nn.Linear(encoder_dim, attn_dim, bias=False)
        self.W_a = nn.Linear(attn_dim, attn_dim, bias=False)
        self.scale = math.sqrt(attn_dim)
        self.output_proj = nn.Linear(attn_dim, decoder_dim)
    def forward(self, query, encoder_output, mask=None):
        Q = self.query_proj(query); K = self.key_proj(encoder_output); V = self.value_proj(encoder_output)
        scores = torch.bmm(self.W_a(Q), K.transpose(1, 2)) / self.scale
        if mask is not None: scores = scores.masked_fill(mask == 0, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        return self.output_proj(torch.bmm(weights, V)), weights

class OCRModel(nn.Module):
    def __init__(self, vocab_size, bos_id=2, eos_id=3, decoder_dim=512, embed_dim=256, attn_dim=256,
                 max_len=128, lstm_hidden=256, lstm_layers=2, dropout=0.3):
        super().__init__()
        self.encoder = CNNBiLSTMEncoder(lstm_hidden, lstm_layers, dropout)
        enc_dim = self.encoder.output_size
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.decoder_lstm = nn.LSTMCell(embed_dim + decoder_dim, decoder_dim)
        self.attention = MultiplicativeAttention(enc_dim, decoder_dim, attn_dim)
        self.init_h = nn.Linear(enc_dim, decoder_dim)
        self.init_c = nn.Linear(enc_dim, decoder_dim)
        self.output_head = nn.Sequential(nn.Linear(decoder_dim, decoder_dim), nn.ReLU(),
                                         nn.Dropout(dropout), nn.Linear(decoder_dim, vocab_size))
        self.dropout = nn.Dropout(dropout)
        self.vocab_size, self.decoder_dim, self.max_len = vocab_size, decoder_dim, max_len
        self.bos_id, self.eos_id = bos_id, eos_id

    def _init_decoder_state(self, encoder_out):
        m = encoder_out.mean(dim=1)
        return torch.tanh(self.init_h(m)), torch.tanh(self.init_c(m))

    def forward(self, pixel_values, labels=None, teacher_forcing_ratio=1.0):
        B = pixel_values.size(0)
        enc = self.encoder(pixel_values)
        h, c = self._init_decoder_state(enc)
        if labels is not None:
            full_seq = labels.clone(); full_seq[full_seq == -100] = 0
            dec_targets = labels[:, 1:].clone()
            seq_len = dec_targets.size(1)
            inp = full_seq[:, 0]
        else:
            seq_len = self.max_len - 1
            inp = torch.full((B,), self.bos_id, dtype=torch.long, device=pixel_values.device)
        all_logits = []
        for t in range(seq_len):
            emb = self.dropout(self.embedding(inp))
            ctx, _ = self.attention(h.unsqueeze(1), enc)
            ctx = ctx.squeeze(1)
            h, c = self.decoder_lstm(torch.cat([emb, ctx], 1), (h, c))
            logit = self.output_head(self.dropout(h))
            all_logits.append(logit)
            if labels is not None and t + 1 < seq_len:
                inp = full_seq[:, t + 1] if random.random() < teacher_forcing_ratio else logit.argmax(-1)
            else:
                inp = logit.argmax(-1)
        logits = torch.stack(all_logits, 1)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.reshape(-1, self.vocab_size), dec_targets.reshape(-1),
                                   ignore_index=-100, label_smoothing=LABEL_SMOOTH)
        return {"loss": loss, "logits": logits}

    @torch.no_grad()
    def generate_greedy(self, pixel_values, max_length=128):
        self.eval()
        B = pixel_values.size(0)
        enc = self.encoder(pixel_values)
        h, c = self._init_decoder_state(enc)
        inp = torch.full((B,), self.bos_id, dtype=torch.long, device=pixel_values.device)
        gen = [inp]
        for _ in range(max_length - 1):
            emb = self.embedding(inp)
            ctx, _ = self.attention(h.unsqueeze(1), enc)
            h, c = self.decoder_lstm(torch.cat([emb, ctx.squeeze(1)], 1), (h, c))
            inp = self.output_head(h).argmax(-1)
            gen.append(inp)
            if (inp == self.eos_id).all(): break
        return torch.stack(gen, 1)

    @torch.no_grad()
    def generate_beam(self, pixel_values, beam_width=5, max_length=128):
        self.eval()
        assert pixel_values.size(0) == 1
        enc = self.encoder(pixel_values)
        h0, c0 = self._init_decoder_state(enc)
        h0, c0 = h0.squeeze(0), c0.squeeze(0)
        beams, complete = [(0.0, [self.bos_id], h0, c0)], []
        for _ in range(max_length - 1):
            candidates = []
            for lp, toks, h, c in beams:
                if toks[-1] == self.eos_id:
                    complete.append((lp / max(len(toks), 1), toks)); continue
                inp = torch.tensor([toks[-1]], device=pixel_values.device)
                emb = self.embedding(inp)
                ctx, _ = self.attention(h.unsqueeze(0).unsqueeze(0), enc)
                nh, nc = self.decoder_lstm(torch.cat([emb, ctx.squeeze(1)], 1), (h.unsqueeze(0), c.unsqueeze(0)))
                nh, nc = nh.squeeze(0), nc.squeeze(0)
                log_p = F.log_softmax(self.output_head(nh.unsqueeze(0)), -1).squeeze(0)
                topv, topi = log_p.topk(beam_width)
                for k in range(beam_width):
                    candidates.append((lp + topv[k].item(), toks + [topi[k].item()], nh, nc))
            if not candidates: break
            candidates.sort(key=lambda x: x[0] / max(len(x[1]), 1), reverse=True)
            beams = candidates[:beam_width]
        for lp, toks, _, _ in beams:
            complete.append((lp / max(len(toks), 1), toks))
        complete.sort(key=lambda x: x[0], reverse=True)
        return torch.tensor([complete[0][1]], device=pixel_values.device)

model = OCRModel(vocab_size=tokenizer.vocab_size, bos_id=2, eos_id=3, decoder_dim=DECODER_DIM,
                 embed_dim=EMBED_DIM, attn_dim=ATTN_DIM, max_len=MAX_LEN,
                 lstm_hidden=LSTM_HIDDEN, lstm_layers=LSTM_LAYERS).to(device)
print("Params:", f"{sum(p.numel() for p in model.parameters()):,}",
      "| trainable:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Params: 31,806,132 | trainable: 30,361,204


In [6]:
# ---------- metrics (no extra packages needed) ----------
def lev(a, b):
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]

def char_f1(ref, hyp):
    if not ref and not hyp: return 1.0
    if not ref or not hyp: return 0.0
    common = sum((Counter(ref) & Counter(hyp)).values())
    if common == 0: return 0.0
    p, r = common / len(hyp), common / len(ref)
    return 2 * p * r / (p + r)

def compute_all_metrics(preds, labels):
    pairs = [(r, p) for r, p in zip(labels, preds) if len(r.strip()) > 0]
    if not pairs: return {"accuracy": 0.0, "cer": 1.0, "wer": 1.0, "f1": 0.0}
    cer = sum(lev(r, p) for r, p in pairs) / max(1, sum(len(r) for r, _ in pairs))
    wer = sum(lev(r.split(), p.split()) for r, p in pairs) / max(1, sum(len(r.split()) for r, _ in pairs))
    return {"cer": cer, "wer": wer,
            "accuracy": sum(r == p for r, p in pairs) / len(pairs),
            "f1": sum(char_f1(r, p) for r, p in pairs) / len(pairs)}

@torch.no_grad()
def evaluate(loader):
    model.eval(); vp, vl = [], []
    for batch in loader:
        ids = model.generate_greedy(batch["pixel_values"].to(device), MAX_LEN)
        for i in range(ids.size(0)):
            vp.append(tokenizer.decode(ids[i].cpu()))
            lb = batch["labels"][i].clone(); lb[lb == -100] = tokenizer.pad_id
            vl.append(tokenizer.decode(lb.cpu()))
    return compute_all_metrics(vp, vl)

# ---------- training helpers (same as the original) ----------
def get_tf_ratio(epoch, num_epochs, warmup=3, end=0.3):
    if epoch < warmup: return 1.0
    progress = (epoch - warmup) / max(1, num_epochs - warmup - 1)
    return 1.0 - (1.0 - end) * min(progress, 1.0)

class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps, total_steps):
        self.opt, self.warmup, self.total, self.step_n = optimizer, warmup_steps, total_steps, 0
        self.base_lrs = [pg["lr"] for pg in optimizer.param_groups]
    def step(self):
        self.step_n += 1
        if self.step_n <= self.warmup: s = self.step_n / self.warmup
        else:
            p = (self.step_n - self.warmup) / max(1, self.total - self.warmup)
            s = 0.5 * (1 + math.cos(math.pi * p))
        for pg, blr in zip(self.opt.param_groups, self.base_lrs):
            pg["lr"] = blr * max(s, 1e-7)
    def get_lr(self): return [pg["lr"] for pg in self.opt.param_groups]

class ResumeSampler(Sampler):
    """Same shuffle for a given epoch, so a mid-epoch resume skips exactly the batches already done."""
    def __init__(self, n, epoch, seed, skip_samples):
        g = torch.Generator(); g.manual_seed(seed + epoch)
        self.idx = torch.randperm(n, generator=g).tolist()[skip_samples:]
    def __iter__(self): return iter(self.idx)
    def __len__(self): return len(self.idx)

def loadable(path):
    """Kaggle unzips uploaded .pth files into folders. Re-zip such a folder so torch.load can read it."""
    if os.path.isfile(path): return path
    base = next((r for r, d, f in os.walk(path) if "data.pkl" in f), None)
    assert base, f"no data.pkl under {path}"
    out = "/kaggle/working/_rezipped_" + os.path.basename(path.rstrip("/")) + ".pth"
    top = os.path.basename(base)
    with zipfile.ZipFile(out, "w", zipfile.ZIP_STORED) as z:
        for r, d, fs in os.walk(base):
            for f in fs:
                full = os.path.join(r, f)
                z.write(full, os.path.join(top, os.path.relpath(full, base)))
    return out
print("helpers ready")

helpers ready


In [7]:
# ===== load weights + SAFETY GATE =====
val_loader = DataLoader(Subset(eval_dataset, list(range(min(200, len(eval_dataset))))),
                        batch_size=EVAL_BATCH, shuffle=False)
RS = None
best_cer = float("inf")
history = {"loss": [], "cer": [], "acc": [], "f1": []}

if RESUME_STATE_PATH:
    RS = torch.load(loadable(RESUME_STATE_PATH), map_location="cpu", weights_only=False)
    model.load_state_dict(RS["model"], strict=True)
    best_cer, history = RS["best_cer"], RS["history"]
    print(f"Resumed own state: next epoch index {RS['epoch']} (= epoch {RS['epoch']+1}), batch {RS['step']}")
else:
    ck = torch.load(loadable(INIT_WEIGHTS), map_location="cpu", weights_only=False)
    if "model_state_dict" in ck: ck = ck["model_state_dict"]
    assert ck["embedding.weight"].shape[0] == tokenizer.vocab_size, "vocab mismatch between checkpoint and tokenizer.json"
    model.load_state_dict(ck, strict=True)
    del ck; gc.collect()
    print("Weights loaded (strict). Running gate on 200 validation images...")
    m0 = evaluate(val_loader)
    print({k: round(v, 4) for k, v in m0.items()}, "| expected about Acc 0.825, CER 0.0145, WER 0.0452, F1 0.9947")
    assert m0["accuracy"] >= GATE_MIN_ACC and m0["cer"] <= GATE_MAX_CER, \
        "GATE FAILED: loaded weights do not reproduce epoch-12 quality. Do not train; send me this printout."
    best_cer = m0["cer"]
    print("Gate passed. Safe to train.")

Weights loaded (strict). Running gate on 200 validation images...
{'cer': 0.0145, 'wer': 0.0452, 'accuracy': 0.825, 'f1': 0.9947} | expected about Acc 0.825, CER 0.0145, WER 0.0452, F1 0.9947
Gate passed. Safe to train.


In [8]:
# ===================== TRAINING =====================
cnn_p, other_p = [], []
for name, p in model.named_parameters():
    if not p.requires_grad: continue
    (cnn_p if "encoder.cnn" in name else other_p).append(p)
optimizer = torch.optim.AdamW([
    {"params": cnn_p,   "lr": CNN_LR,        "weight_decay": 0.01},
    {"params": other_p, "lr": LEARNING_RATE, "weight_decay": 0.01},
])
opt_per_epoch = steps_per_epoch // GRAD_ACCUM
total_steps = opt_per_epoch * NUM_EPOCHS
scheduler = WarmupCosineScheduler(optimizer, WARMUP_STEPS, total_steps)   # built before loading optimizer state
scaler = torch.amp.GradScaler("cuda")

if RS is not None:
    optimizer.load_state_dict(RS["optimizer"])
    if RS.get("scaler"): scaler.load_state_dict(RS["scaler"])
    scheduler.step_n = RS["sched_step"]
    start_epoch, start_step = RS["epoch"], RS["step"]
    carry_loss, carry_n = RS["ep_loss"], RS["n_batch"]
    rewarm_done = REWARM_STEPS            # optimizer state restored, no re-warmup needed
else:
    start_epoch, start_step = START_EPOCH, 0
    scheduler.step_n = START_EPOCH * opt_per_epoch     # 12 * 2250 = 27000
    carry_loss, carry_n = 0.0, 0
    rewarm_done = 0
print("LR at resume point:", [f"{x:.1e}" for x in (lambda: [blr * (0.5*(1+math.cos(math.pi*(scheduler.step_n-WARMUP_STEPS)/(total_steps-WARMUP_STEPS)))) for blr in scheduler.base_lrs])()],
      "(your log at epoch 13 start: 3.5e-06 and 1.1e-04)")
print(f"Starting at epoch {start_epoch+1}/{NUM_EPOCHS}, batch {start_step}/{steps_per_epoch}, tf={get_tf_ratio(start_epoch, NUM_EPOCHS, SS_WARMUP_EPOCHS, SS_END_RATIO):.2f}")

STATE_PATH = os.path.join(OUTPUT_DIR, "resume_state.pt")
def save_state(next_epoch, next_step, ep_loss, n_batch):
    st = {"model": model.state_dict(), "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
          "sched_step": scheduler.step_n, "epoch": next_epoch, "step": next_step,
          "ep_loss": ep_loss, "n_batch": n_batch, "best_cer": best_cer, "history": history}
    torch.save(st, STATE_PATH + ".tmp"); os.replace(STATE_PATH + ".tmp", STATE_PATH)

last_ep_hours = 0.95
for epoch in range(start_epoch, NUM_EPOCHS):
    if epoch > start_epoch and (time.time() - T0) / 3600 + last_ep_hours > TIME_LIMIT_HOURS:
        print(f"Stopping before epoch {epoch+1}: it would not fit in {TIME_LIMIT_HOURS} h. Resume from resume_state.pt.")
        break
    ep_t0 = time.time()
    skip = start_step if epoch == start_epoch else 0
    ep_loss, n_batch = (carry_loss, carry_n) if (epoch == start_epoch and RS is not None) else (0.0, 0)
    loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=ResumeSampler(len(train_dataset), epoch, SEED, skip * BATCH_SIZE),
                        num_workers=2, pin_memory=True, drop_last=True)
    random.seed(SEED + epoch)
    model.train()
    tf = get_tf_ratio(epoch, NUM_EPOCHS, SS_WARMUP_EPOCHS, SS_END_RATIO)
    optimizer.zero_grad()

    for i, batch in enumerate(loader):
        step = skip + i
        pv = batch["pixel_values"].to(device, non_blocking=True)
        lb = batch["labels"].to(device, non_blocking=True)
        try:
            with torch.amp.autocast("cuda"):
                out = model(pv, lb, tf)
                loss = out["loss"] / GRAD_ACCUM
            if not torch.isfinite(loss):
                print(f"  Skip NaN/Inf step {step}"); optimizer.zero_grad(); continue
            scaler.scale(loss).backward()
            if (step + 1) % GRAD_ACCUM == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad()
                scheduler.step()
                if rewarm_done < REWARM_STEPS:
                    rewarm_done += 1
                    f = rewarm_done / REWARM_STEPS
                    for pg in optimizer.param_groups: pg["lr"] *= f
            ep_loss += loss.item() * GRAD_ACCUM; n_batch += 1
            if step % 500 == 0:
                lrs = ", ".join(f"{x:.1e}" for x in scheduler.get_lr())
                print(f"{time.time()-T0:8.1f}s  E{epoch+1} S{step}/{steps_per_epoch} loss={loss.item()*GRAD_ACCUM:.4f} lr=[{lrs}] tf={tf:.2f}")
            if (step + 1) % SAVE_EVERY_STEPS == 0 and (step + 1) % GRAD_ACCUM == 0 and step + 1 < steps_per_epoch:
                save_state(epoch, step + 1, ep_loss, n_batch)
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"  OOM step {step}, skipping"); torch.cuda.empty_cache(); optimizer.zero_grad(); continue
            raise

    avg_loss = ep_loss / max(n_batch, 1)
    history["loss"].append(avg_loss)
    m = evaluate(val_loader)
    history["cer"].append(m["cer"]); history["acc"].append(m["accuracy"]); history["f1"].append(m["f1"])
    tag = ""
    if m["cer"] < best_cer:
        best_cer = m["cer"]
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "best_model.pth"))
        tokenizer.save(os.path.join(OUTPUT_DIR, "tokenizer.json"))
        tag = " ** BEST **"
    torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "latest_model.pth"))
    save_state(epoch + 1, 0, 0.0, 0)
    last_ep_hours = (time.time() - ep_t0) / 3600
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | loss={avg_loss:.4f} | CER={m['cer']:.4f} WER={m['wer']:.4f} "
          f"F1={m['f1']:.4f} Acc={m['accuracy']:.4f}{tag} | {last_ep_hours:.2f} h")
print(f"Done. Best CER: {best_cer:.4f}")

LR at resume point: ['3.5e-06', '1.1e-04'] (your log at epoch 13 start: 3.5e-06 and 1.1e-04)
Starting at epoch 13/20, batch 0/9000, tf=0.61
   231.1s  E13 S0/9000 loss=0.9759 lr=[1.0e-05, 3.0e-04] tf=0.61
   417.7s  E13 S500/9000 loss=0.8420 lr=[3.5e-06, 1.0e-04] tf=0.61
   603.7s  E13 S1000/9000 loss=0.8261 lr=[3.4e-06, 1.0e-04] tf=0.61
   790.1s  E13 S1500/9000 loss=1.2236 lr=[3.4e-06, 1.0e-04] tf=0.61
   976.7s  E13 S2000/9000 loss=0.8448 lr=[3.4e-06, 1.0e-04] tf=0.61
  1162.4s  E13 S2500/9000 loss=1.4363 lr=[3.3e-06, 9.9e-05] tf=0.61
  1350.5s  E13 S3000/9000 loss=0.8492 lr=[3.3e-06, 9.8e-05] tf=0.61
  1538.3s  E13 S3500/9000 loss=1.0348 lr=[3.2e-06, 9.7e-05] tf=0.61
  1725.8s  E13 S4000/9000 loss=0.8714 lr=[3.2e-06, 9.6e-05] tf=0.61
  1910.8s  E13 S4500/9000 loss=0.8546 lr=[3.1e-06, 9.4e-05] tf=0.61
  2097.6s  E13 S5000/9000 loss=0.8735 lr=[3.1e-06, 9.3e-05] tf=0.61
  2282.9s  E13 S5500/9000 loss=0.8594 lr=[3.1e-06, 9.2e-05] tf=0.61
  2469.0s  E13 S6000/9000 loss=0.8764 lr=[3.0e-0

In [9]:
# ===== final evaluation: greedy vs beam on the first EVAL_SAMPLES test images =====
tokenizer.save(os.path.join(OUTPUT_DIR, "tokenizer.json"))
bp = os.path.join(OUTPUT_DIR, "best_model.pth")
if os.path.exists(bp):
    model.load_state_dict(torch.load(bp, map_location=device)); print("Loaded best model")
model.eval()
n = min(EVAL_SAMPLES, len(eval_dataset))
pg, pb, ref = [], [], []
for i in range(n):
    s = eval_dataset[i]
    pv = s["pixel_values"].unsqueeze(0).to(device)
    pg.append(tokenizer.decode(model.generate_greedy(pv, MAX_LEN)[0].cpu()))
    pb.append(tokenizer.decode(model.generate_beam(pv, BEAM_WIDTH, MAX_LEN)[0].cpu()))
    lb = s["labels"].clone(); lb[lb == -100] = tokenizer.pad_id
    ref.append(tokenizer.decode(lb))
mg, mb = compute_all_metrics(pg, ref), compute_all_metrics(pb, ref)
print(f"{'Metric':<10}{'Greedy':>10}{'Beam-'+str(BEAM_WIDTH):>10}")
for k in ["cer", "wer", "f1", "accuracy"]:
    print(f"{k.upper():<10}{mg[k]:>10.4f}{mb[k]:>10.4f}")

Loaded best model
Metric        Greedy    Beam-5
CER           0.0081    0.0056
WER           0.0295    0.0216
F1            0.9966    0.9976
ACCURACY      0.9050    0.9200
